In [1]:
import os
import sys
import glob
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 50)
pd.set_option('display.max_colwidth', 150)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns
from math import log
from scipy import stats
sns.set_theme()
try:
    from cellacdc import cca_functions
    from cellacdc import myutils
except FileNotFoundError:
    # Check if user has developer version --> add the Cell_ACDC/cellacdc
    # folder to path and import from thre
    sys.path.insert(0, '../cellacdc/')
    from cellacdc import cca_functions
    from cellacdc import myutils

# ***Import nutrient switch data - three reps - and concatenate*** 
# ***-update f_pos_cell_id with additional rep***

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 400)


rep1_rel_switch_wave_df= pd.read_csv(r"Y:\test_dfs_for_final_code\09082022_jupyter_overall_df_with_rel_switch_and_wave_info.csv")
rep1_rel_switch_wave_df['rep'] = 1
rep1_rel_switch_wave_df = rep1_rel_switch_wave_df.drop(['Unnamed: 0', 'Dia_Ph3_combine_metrics_example', 'Dia_Ph3_combine_metrics_example_rel'], axis=1)
print(rep1_rel_switch_wave_df.shape)
#display(rep1_rel_switch_wave_df.head(5))

rep2_rel_switch_wave_df= pd.read_csv(r"Y:\test_dfs_for_final_code\13102022_jupyter_overall_df_with_rel_switch_and_wave_info.csv")
rep2_rel_switch_wave_df['rep'] = 2
rep2_rel_switch_wave_df = rep2_rel_switch_wave_df.drop(['Unnamed: 0'], axis=1)
print(rep2_rel_switch_wave_df.shape) 
#display(rep2_rel_switch_wave_df.head(5))

rep3_rel_switch_wave_df= pd.read_csv(r"Y:\test_dfs_for_final_code\28032023_jupyter_overall_df_with_rel_switch_and_wave_info.csv")
rep3_rel_switch_wave_df['rep'] = 3
rep3_rel_switch_wave_df = rep3_rel_switch_wave_df.drop(['Unnamed: 0'], axis=1)
print(rep3_rel_switch_wave_df.shape) 
a =set((rep3_rel_switch_wave_df.columns).difference(rep2_rel_switch_wave_df.columns))
print(a)
rep3_rel_switch_wave_df.drop(columns = a, inplace = True)
print(rep3_rel_switch_wave_df.shape)
#display(rep2_rel_switch_wave_df.head(5))

rep_df_list = [rep1_rel_switch_wave_df, rep2_rel_switch_wave_df, rep3_rel_switch_wave_df]
overall_df_with_rel_and_switch_info = pd.concat(rep_df_list).reset_index(drop = True)
print(overall_df_with_rel_and_switch_info.shape)
overall_df_with_rel_and_switch_info['rep_str_pos_cell_id'] = overall_df_with_rel_and_switch_info.apply(lambda x: f'{x["f_pos_cell_id"]}_Rep_{int(x["rep"])}', axis=1)
overall_df_with_rel_and_switch_info = overall_df_with_rel_and_switch_info.drop(['f_pos_cell_id'], axis=1)
overall_df_with_rel_and_switch_info = overall_df_with_rel_and_switch_info.rename(columns={"rep_str_pos_cell_id": "f_pos_cell_id"})
#display(overall_df_with_rel_and_switch_info.head(5))
print(overall_df_with_rel_and_switch_info.rep.unique())

C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_16056\3645002057.py:6: DtypeWarning: Columns (373) have mixed types. Specify dtype option on import or set low_memory=False.
  rep1_rel_switch_wave_df= pd.read_csv(r"Y:\test_dfs_for_final_code\09082022_jupyter_overall_df_with_rel_switch_and_wave_info.csv")


(155801, 390)


C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_16056\3645002057.py:12: DtypeWarning: Columns (321) have mixed types. Specify dtype option on import or set low_memory=False.
  rep2_rel_switch_wave_df= pd.read_csv(r"Y:\test_dfs_for_final_code\13102022_jupyter_overall_df_with_rel_switch_and_wave_info.csv")


(144734, 340)
(182028, 362)
{'local_centroid_rel', 'centroid_rel', 'moments_normalized', 'moments_normalized_rel', 'inertia_tensor_rel', 'moments_rel', 'time_hours_rel', 'moments', 'moments_central_rel', 'inertia_tensor', 'label_rel', 'inertia_tensor_eigvals', 'inertia_tensor_eigvals_rel', 'local_centroid', 'time_hours', 'time_minutes_rel', 'moments_central', 'time_minutes', 'bbox', 'label', 'bbox_rel', 'centroid'}
(182028, 340)
(482563, 390)
[1 2 3]


# ***Import steady state data - both media*** 
# ***-assign complete_phase = False to some faultily annotated cells***

In [5]:
df = pd.read_csv(r"Y:\test_dfs_for_final_code\complete_optimised4MLR_SCD_SCGE_merged_dataset.csv")
df.drop(df[(df.Position == "Position_23") & (df.date == 21012021)].index, inplace=True)
df.drop(df[(df.Position == "Position_15") & (df.date == 20012021)].index, inplace=True)

df['complete_phase'].loc[df[(df['growthmedium'] == 'SCGE') & (df['date'] == 23092021) & (df['Position'] == 'Position_8') & (df['Cell_ID'] == 2)].index] = False
#last cell cycle stage annotated wrong:
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 20012021) & (df['Position'] == 'Position_33') & (df['Cell_ID'] == 32)].index] = False
#first generation number annotated wrong:
df['generation_num'].loc[df[(df['growthmedium'] == 'SCGE') & (df['Position'] == 'Position_8') & (df['Cell_ID'] == 2) & (df['generation_num'] == 0)].index] = 2
#relatives missing in last frame or missing from data altogether, therefore no system volume at last frame, therefore abs_growth_rate = Nan.
# Labelling both G1 and S phases in these generations with missing relatives as incomplete phases because we have incomplete information
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCGE') & (df['Position'] == 'Position_27') & (df['Cell_ID'] == 48) & (df['generation_num'] == 1)].index] = False
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCGE') & (df['Position'] == 'Position_38') & (df['Cell_ID'] == 25) & (df['generation_num'] == 1)].index] = False
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 21012021) & (df['Position'] == 'Position_12') & (df['Cell_ID'] == 3) & (df['generation_num'] == 3)].index] = False
# for this cell, the last frame of generation 1 G1 is labelled as generation 2 G1. correcting to generation_num = 1 would not be sufficient,
# as all the other columns for this frame are calculated as per S phase. Therefore, labelling all phases in generation 1 for this cell incomplete
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 20012021) & (df['Position'] == 'Position_20') & (df['Cell_ID'] == 16)
                            & ((df['generation_num'] == 1)|(df['generation_num'] == 2))].index] = False
# gen 4 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen4 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 21012021) & (df['Position'] == 'Position_1') & (df['Cell_ID'] == 2)
                            & (df['generation_num'] == 4)].index] = False
# gen 4 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen4 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 20012021) & (df['Position'] == 'Position_11') & (df['Cell_ID'] == 3)
                            & (df['generation_num'] == 4)].index] = False
#len_g1 value missing from gen 1 for some reason. remove gen 1 completely:
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 21012021) & (df['Position'] == 'Position_7') & (df['Cell_ID'] == 17)
                            & (df['generation_num'] == 1)].index] = False
# gen 4 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen4 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCGE') & (df['date'] == 23092021) & (df['Position'] == 'Position_24') & (df['Cell_ID'] == 1)
                            & (df['generation_num'] == 4)].index] = False
# gen 3 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen3 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 20012021) & (df['Position'] == 'Position_20') & (df['Cell_ID'] == 7)
                            & (df['generation_num'] == 3)].index] = False
# gen 3 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen3 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 21012021) & (df['Position'] == 'Position_28') & (df['Cell_ID'] == 7)
                            & (df['generation_num'] == 3)].index] = False
# gen 5 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen5 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 21012021) & (df['Position'] == 'Position_29') & (df['Cell_ID'] == 3)
                            & (df['generation_num'] == 5)].index] = False
# gen 4 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen4 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 21012021) & (df['Position'] == 'Position_15') & (df['Cell_ID'] == 3)
                            & (df['generation_num'] == 4)].index] = False
# gen 5 skips G1 phase and goes directly from S to S. therefore no len_G1 value. removing gen5 completely
df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 21012021) & (df['Position'] == 'Position_15') & (df['Cell_ID'] == 5)
                            & (df['generation_num'] == 5)].index] = False




df['sys_vol_at_frame_inc_G1_cells'] = df['sys_vol_atframe'].combine_first(df['cell_vol_fl'])




C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_16056\1886980860.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['complete_phase'].loc[df[(df['growthmedium'] == 'SCGE') & (df['date'] == 23092021) & (df['Position'] == 'Position_8') & (df['Cell_ID'] == 2)].index] = False
C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_16056\1886980860.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['complete_phase'].loc[df[(df['growthmedium'] == 'SCD') & (df['date'] == 20012021) & (df['Position'] == 'Position_33') & (df['Cell_ID'] == 32)].index] = False
C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_16056\18869808

# ***Rename nutrient switch df to nutrient_switch_overall_df*** 
# ***Rename steady state df to steady_state_overall_df***

In [6]:
nutrient_switch_overall_df = overall_df_with_rel_and_switch_info
steady_state_overall_df = df

# ***assign complete_cycle to steady state data*** 

In [7]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 400)

steady_state_overall_df['f_pos_cell_id'] = steady_state_overall_df.apply(
                                            lambda x: f'{x["growthmedium"]}_{x["Strain"]}_{x["Position"]}_Cell_{x["Cell_ID"]}_Gen_{int(x["generation_num"])}_Date_{int(x["date"])}', axis=1)


for fposid in ((steady_state_overall_df['f_pos_cell_id']).unique()):
    fposid_df = steady_state_overall_df[steady_state_overall_df['f_pos_cell_id'] == fposid]
    fposid_df = fposid_df.sort_values(by=['frame_i'])
    if (len(fposid_df['cell_cycle_stage'].unique())) == 1:
        steady_state_overall_df.loc[(steady_state_overall_df['f_pos_cell_id'] == fposid), 'complete_cycle'] = 0
    else:
        if len(fposid_df['complete_phase'].unique()) > 1:
            steady_state_overall_df.loc[(steady_state_overall_df['f_pos_cell_id'] == fposid), 'complete_cycle'] = 0
        elif ((len(fposid_df['complete_phase'].unique()) == 1) & ((fposid_df['complete_phase'].unique()[0]) == False)):
            steady_state_overall_df.loc[(steady_state_overall_df['f_pos_cell_id'] == fposid), 'complete_cycle'] = 0
        elif ((len(fposid_df['complete_phase'].unique()) == 1) & ((fposid_df['complete_phase'].unique()[0]) == True)):
            steady_state_overall_df.loc[(steady_state_overall_df['f_pos_cell_id'] == fposid), 'complete_cycle'] = 1    
            



,Unnamed: 0.1,index,Unnamed: 0,generation_num,Position,Cell_ID,growthmedium,frame_i,cell_cycle_stage,relative_ID,relationship,cell_vol_fl,emerg_frame_i,division_frame_i,len_G1,date,Strain,Birth frame,Birth vol fl,last_G1_frame,size_G1_end,vol_added_G1,Div frame,Div vol fl,len_S,mother_size_emerg,mother_size_div,vol_added_S_mother,bud_size_emerg,bud_size_div,vol_added_S_bud,sys_size_emerg,sys_size_div,vol_added_S_tot,total vol added,time_seconds,is_cell_dead,complete_phase,lenG1_mins,G1_growthrate_flpermin,delta_vol_fl,delta_time_mins,abs_growth_rate,Birth_or_div_vol_fl,rel_vol_atframe,sys_vol_atframe,delta_vol_sincePhaseStart_mother,delta_vol_sincePhaseStart_sys,norm_delta_vol_sincePhaseStart_mother,norm_delta_vol_sincePhaseStart_sys,bud_vol_atframe/mother_vol_atframe,delta_vol_sinceG1Start,norm_delta_vol_sinceG1Start,Daughterhood_binary,time_in_stage_mins,sys_vol_at_frame_inc_G1_cells,f_pos_cell_id
0,0,0.0,0.0,0.0,Position_1,1,SCD,0,S,4.0,bud,58.256695,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,False,NaN,NaN,NaN,18,NaN,NaN,92.927348,151.184043,NaN,NaN,NaN,NaN,1.595136,NaN,NaN,0,0.0,151.184043,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021
1,1,1.0,1.0,0.0,Position_1,1,SCD,1,S,4.0,bud,60.404339,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.0,0,False,NaN,NaN,NaN,18,NaN,NaN,93.706004,154.110342,NaN,NaN,NaN,NaN,1.551312,NaN,NaN,0,3.0,154.110342,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021
2,2,2.0,2.0,0.0,Position_1,1,SCD,2,S,4.0,bud,61.139191,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,360.0,0,False,NaN,NaN,NaN,18,NaN,NaN,94.112864,155.252054,NaN,NaN,NaN,NaN,1.539321,NaN,NaN,0,6.0,155.252054,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021
3,3,3.0,3.0,0.0,Position_1,1,SCD,3,S,4.0,bud,62.500215,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,540.0,0,False,NaN,NaN,NaN,18,NaN,NaN,95.679685,158.179900,NaN,NaN,NaN,NaN,1.530870,NaN,NaN,0,9.0,158.179900,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021
4,4,4.0,4.0,0.0,Position_1,1,SCD,4,S,4.0,bud,63.211776,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,720.0,0,False,NaN,NaN,NaN,18,NaN,NaN,97.265445,160.477221,NaN,NaN,NaN,NaN,1.538723,NaN,NaN,0,12.0,160.477221,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021
5,5,5.0,5.0,0.0,Position_1,1,SCD,5,S,4.0,bud,63.341337,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,900.0,0,False,NaN,NaN,NaN,18,NaN,NaN,99.942398,163.283735,NaN,NaN,NaN,NaN,1.577838,NaN,NaN,0,15.0,163.283735,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021
6,6,6.0,6.0,0.0,Position_1,1,SCD,6,S,4.0,bud,64.975784,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1080.0,0,False,NaN,NaN,NaN,18,NaN,NaN,102.084921,167.060705,NaN,NaN,NaN,NaN,1.571123,NaN,NaN,0,18.0,167.060705,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021
7,7,7.0,7.0,1.0,Position_1,1,SCD,7,G1,4.0,mother,67.499547,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1260.0,0,True,39.0,0.496112,71.738164,111,0.64629,67.499547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1,0.0,67.499547,SCD_MMY_Position_1_Cell_1_Gen_1_Date_20012021
8,8,8.0,8.0,1.0,Position_1,1,SCD,8,G1,4.0,mother,67.411987,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1440.0,0,True,39.0,0.496112,71.738164,111,0.64629,67.499547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.087560,-0.001297,1,3.0,67.411987,SCD_MMY_Position_1_Cell_1_Gen_1_Date_20012021
9,9,9.0,9.0,1.0,Position_1,1,SCD,9,G1,4.0,mother,68.835553,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.34

[False  True]
[0. 1.]


37.725596345943856

# ***Assigning frames since cycle start for steady state data*** 

In [8]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 400)

import warnings

with warnings.catch_warnings(record=True):
    unique_f_pos_cell_id = []
    for fposid in ((steady_state_overall_df['f_pos_cell_id']).unique()):
        fposid_df = steady_state_overall_df[steady_state_overall_df['f_pos_cell_id'] == fposid]
        fposid_df = fposid_df.sort_values(by=['frame_i'])
        cycle_start_frame =fposid_df['frame_i'].iloc[0]
        fposid_df['frames_since_cycle_start'] = np.nan
        for row in fposid_df.index:
            fposid_df['frames_since_cycle_start'].loc[row] = (fposid_df['frame_i'].loc[row]) - cycle_start_frame 
        unique_f_pos_cell_id.append(fposid_df)
    steady_state_overall_df = pd.concat(unique_f_pos_cell_id)
# display(steady_state_overall_df.head(10))

,Unnamed: 0.1,index,Unnamed: 0,generation_num,Position,Cell_ID,growthmedium,frame_i,cell_cycle_stage,relative_ID,relationship,cell_vol_fl,emerg_frame_i,division_frame_i,len_G1,date,Strain,Birth frame,Birth vol fl,last_G1_frame,size_G1_end,vol_added_G1,Div frame,Div vol fl,len_S,mother_size_emerg,mother_size_div,vol_added_S_mother,bud_size_emerg,bud_size_div,vol_added_S_bud,sys_size_emerg,sys_size_div,vol_added_S_tot,total vol added,time_seconds,is_cell_dead,complete_phase,lenG1_mins,G1_growthrate_flpermin,delta_vol_fl,delta_time_mins,abs_growth_rate,Birth_or_div_vol_fl,rel_vol_atframe,sys_vol_atframe,delta_vol_sincePhaseStart_mother,delta_vol_sincePhaseStart_sys,norm_delta_vol_sincePhaseStart_mother,norm_delta_vol_sincePhaseStart_sys,bud_vol_atframe/mother_vol_atframe,delta_vol_sinceG1Start,norm_delta_vol_sinceG1Start,Daughterhood_binary,time_in_stage_mins,sys_vol_at_frame_inc_G1_cells,f_pos_cell_id,complete_cycle,frames_since_cycle_start
0,0,0.0,0.0,0.0,Position_1,1,SCD,0,S,4.0,bud,58.256695,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,False,NaN,NaN,NaN,18,NaN,NaN,92.927348,151.184043,NaN,NaN,NaN,NaN,1.595136,NaN,NaN,0,0.0,151.184043,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,0.0
1,1,1.0,1.0,0.0,Position_1,1,SCD,1,S,4.0,bud,60.404339,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.0,0,False,NaN,NaN,NaN,18,NaN,NaN,93.706004,154.110342,NaN,NaN,NaN,NaN,1.551312,NaN,NaN,0,3.0,154.110342,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,1.0
2,2,2.0,2.0,0.0,Position_1,1,SCD,2,S,4.0,bud,61.139191,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,360.0,0,False,NaN,NaN,NaN,18,NaN,NaN,94.112864,155.252054,NaN,NaN,NaN,NaN,1.539321,NaN,NaN,0,6.0,155.252054,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,2.0
3,3,3.0,3.0,0.0,Position_1,1,SCD,3,S,4.0,bud,62.500215,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,540.0,0,False,NaN,NaN,NaN,18,NaN,NaN,95.679685,158.179900,NaN,NaN,NaN,NaN,1.530870,NaN,NaN,0,9.0,158.179900,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,3.0
4,4,4.0,4.0,0.0,Position_1,1,SCD,4,S,4.0,bud,63.211776,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,720.0,0,False,NaN,NaN,NaN,18,NaN,NaN,97.265445,160.477221,NaN,NaN,NaN,NaN,1.538723,NaN,NaN,0,12.0,160.477221,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,4.0
5,5,5.0,5.0,0.0,Position_1,1,SCD,5,S,4.0,bud,63.341337,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,900.0,0,False,NaN,NaN,NaN,18,NaN,NaN,99.942398,163.283735,NaN,NaN,NaN,NaN,1.577838,NaN,NaN,0,15.0,163.283735,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,5.0
6,6,6.0,6.0,0.0,Position_1,1,SCD,6,S,4.0,bud,64.975784,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1080.0,0,False,NaN,NaN,NaN,18,NaN,NaN,102.084921,167.060705,NaN,NaN,NaN,NaN,1.571123,NaN,NaN,0,18.0,167.060705,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,6.0
7,7,7.0,7.0,1.0,Position_1,1,SCD,7,G1,4.0,mother,67.499547,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1260.0,0,True,39.0,0.496112,71.738164,111,0.64629,67.499547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1,0.0,67.499547,SCD_MMY_Position_1_Cell_1_Gen_1_Date_20012021,1.0,0.0
8,8,8.0,8.0,1.0,Position_1,1,SCD,8,G1,4.0,mother,67.411987,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1440.0,0,True,39.0,0.496112,71.738164,111,0.64629,67.499547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.087560,-0.001297,1,3.0,67.411987,SCD_MMY_Position_1_Cell_1_Gen_1_Date_20012021,1.0,1.0
9,9,9.0,

# ***Assigning delta vol to cell vol for both data*** 

In [9]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 400)


import warnings

with warnings.catch_warnings(record=True):
    #################################### for steady state data ######################################################
    unique_f_pos_cell_id = []
    for fposid in ((steady_state_overall_df['f_pos_cell_id']).unique()):
        fposid_df = steady_state_overall_df[steady_state_overall_df['f_pos_cell_id'] == fposid]
        fposid_df = fposid_df.sort_values(by=['frame_i'])
        fposid_df['vol_added_since_last_frame'] = fposid_df['sys_vol_at_frame_inc_G1_cells'].diff()
        unique_f_pos_cell_id.append(fposid_df)
    steady_state_overall_df = pd.concat(unique_f_pos_cell_id)
    #################################### for nutrient switch data ######################################################
    unique_f_pos_cell_id_2 = []
    for fposid in ((nutrient_switch_overall_df['f_pos_cell_id']).unique()):
        fposid_df = nutrient_switch_overall_df[nutrient_switch_overall_df['f_pos_cell_id'] == fposid]
        fposid_df = fposid_df.sort_values(by=['frame_i'])
        fposid_df['vol_added_since_last_frame'] = fposid_df['combined_mother_bud_volume'].diff()
        unique_f_pos_cell_id_2.append(fposid_df)
    nutrient_switch_overall_df = pd.concat(unique_f_pos_cell_id_2)
# display(steady_state_overall_df.head(10))
# display(nutrient_switch_overall_df.head(10))



,Unnamed: 0.1,index,Unnamed: 0,generation_num,Position,Cell_ID,growthmedium,frame_i,cell_cycle_stage,relative_ID,relationship,cell_vol_fl,emerg_frame_i,division_frame_i,len_G1,date,Strain,Birth frame,Birth vol fl,last_G1_frame,size_G1_end,vol_added_G1,Div frame,Div vol fl,len_S,mother_size_emerg,mother_size_div,vol_added_S_mother,bud_size_emerg,bud_size_div,vol_added_S_bud,sys_size_emerg,sys_size_div,vol_added_S_tot,total vol added,time_seconds,is_cell_dead,complete_phase,lenG1_mins,G1_growthrate_flpermin,delta_vol_fl,delta_time_mins,abs_growth_rate,Birth_or_div_vol_fl,rel_vol_atframe,sys_vol_atframe,delta_vol_sincePhaseStart_mother,delta_vol_sincePhaseStart_sys,norm_delta_vol_sincePhaseStart_mother,norm_delta_vol_sincePhaseStart_sys,bud_vol_atframe/mother_vol_atframe,delta_vol_sinceG1Start,norm_delta_vol_sinceG1Start,Daughterhood_binary,time_in_stage_mins,sys_vol_at_frame_inc_G1_cells,f_pos_cell_id,complete_cycle,frames_since_cycle_start,vol_added_since_last_frame
0,0,0.0,0.0,0.0,Position_1,1,SCD,0,S,4.0,bud,58.256695,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,False,NaN,NaN,NaN,18,NaN,NaN,92.927348,151.184043,NaN,NaN,NaN,NaN,1.595136,NaN,NaN,0,0.0,151.184043,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,0.0,NaN
1,1,1.0,1.0,0.0,Position_1,1,SCD,1,S,4.0,bud,60.404339,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.0,0,False,NaN,NaN,NaN,18,NaN,NaN,93.706004,154.110342,NaN,NaN,NaN,NaN,1.551312,NaN,NaN,0,3.0,154.110342,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,1.0,2.926300
2,2,2.0,2.0,0.0,Position_1,1,SCD,2,S,4.0,bud,61.139191,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,360.0,0,False,NaN,NaN,NaN,18,NaN,NaN,94.112864,155.252054,NaN,NaN,NaN,NaN,1.539321,NaN,NaN,0,6.0,155.252054,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,2.0,1.141712
3,3,3.0,3.0,0.0,Position_1,1,SCD,3,S,4.0,bud,62.500215,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,540.0,0,False,NaN,NaN,NaN,18,NaN,NaN,95.679685,158.179900,NaN,NaN,NaN,NaN,1.530870,NaN,NaN,0,9.0,158.179900,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,3.0,2.927846
4,4,4.0,4.0,0.0,Position_1,1,SCD,4,S,4.0,bud,63.211776,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,720.0,0,False,NaN,NaN,NaN,18,NaN,NaN,97.265445,160.477221,NaN,NaN,NaN,NaN,1.538723,NaN,NaN,0,12.0,160.477221,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,4.0,2.297321
5,5,5.0,5.0,0.0,Position_1,1,SCD,5,S,4.0,bud,63.341337,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,900.0,0,False,NaN,NaN,NaN,18,NaN,NaN,99.942398,163.283735,NaN,NaN,NaN,NaN,1.577838,NaN,NaN,0,15.0,163.283735,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,5.0,2.806514
6,6,6.0,6.0,0.0,Position_1,1,SCD,6,S,4.0,bud,64.975784,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1080.0,0,False,NaN,NaN,NaN,18,NaN,NaN,102.084921,167.060705,NaN,NaN,NaN,NaN,1.571123,NaN,NaN,0,18.0,167.060705,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,6.0,3.776970
7,7,7.0,7.0,1.0,Position_1,1,SCD,7,G1,4.0,mother,67.499547,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1260.0,0,True,39.0,0.496112,71.738164,111,0.64629,67.499547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1,0.0,67.499547,SCD_MMY_Position_1_Cell_1_Gen_1_Date_20012021,1.0,0.0,NaN
8,8,8.0,8.0,1.0,Position_1,1,SCD,8,G1,4.0,mother,67.411987,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1440.0,0,True,39.0,0.496112,71.738164,111,0.64629,67.499547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.087560

,frame_i,time_seconds,Cell_ID,is_cell_dead,is_cell_excluded,x_centroid,y_centroid,was_manually_edited,cell_cycle_stage,generation_num,relative_ID,relationship,emerg_frame_i,division_frame_i,is_history_known,corrected_assignment,cell_area_pxl,cell_area_um2,cell_vol_vox,cell_vol_fl,Dia_Ph3_mean,Dia_Ph3_sum,Dia_Ph3_amount_autoBkgr,Dia_Ph3_amount_dataPrepBkgr,Dia_Ph3_concentration_autoBkgr_from_vol_vox,Dia_Ph3_concentration_dataPrepBkgr_from_vol_vox,Dia_Ph3_concentration_autoBkgr_from_vol_fl,Dia_Ph3_concentration_dataPrepBkgr_from_vol_fl,Dia_Ph3_median,Dia_Ph3_min,Dia_Ph3_max,Dia_Ph3_q25,Dia_Ph3_q75,Dia_Ph3_q05,Dia_Ph3_q95,Dia_Ph3_autoBkgr_bkgrVal_median,Dia_Ph3_autoBkgr_bkgrVal_mean,Dia_Ph3_autoBkgr_bkgrVal_q75,Dia_Ph3_autoBkgr_bkgrVal_q25,Dia_Ph3_autoBkgr_bkgrVal_q95,Dia_Ph3_autoBkgr_bkgrVal_q05,Dia_Ph3_dataPrepBkgr_bkgrVal_median,Dia_Ph3_dataPrepBkgr_bkgrVal_mean,Dia_Ph3_dataPrepBkgr_bkgrVal_q75,Dia_Ph3_dataPrepBkgr_bkgrVal_q25,Dia_Ph3_dataPrepBkgr_bkgrVal_q95,Dia_Ph3_dataPrepBkgr_bkgrVal_q05,Dia_Ph3_CV,major_axis_length_gui,minor_axis_length_gui,inertia_tensor_eigvals-0,inertia_tensor_eigvals-1,equivalent_diameter,moments-0-0,moments-0-1,moments-0-2,moments-0-3,moments-1-0,moments-1-1,moments-1-2,moments-1-3,moments-2-0,moments-2-1,moments-2-2,moments-2-3,moments-3-0,moments-3-1,moments-3-2,moments-3-3,area_gui,solidity_gui,extent,inertia_tensor-0-0,inertia_tensor-0-1,inertia_tensor-1-0,inertia_tensor-1-1,filled_area_gui,centroid-0,centroid-1,bbox_area,local_centroid-0,local_centroid-1,convex_area_gui,euler_number,moments_normalized-0-0,moments_normalized-0-1,moments_normalized-0-2,moments_normalized-0-3,moments_normalized-1-0,moments_normalized-1-1,moments_normalized-1-2,moments_normalized-1-3,moments_normalized-2-0,moments_normalized-2-1,moments_normalized-2-2,moments_normalized-2-3,moments_normalized-3-0,moments_normalized-3-1,moments_normalized-3-2,moments_normalized-3-3,moments_central-0-0,moments_central-0-1,moments_central-0-2,moments_central-0-3,moments_central-1-0,moments_central-1-1,moments_central-1-2,moments_central-1-3,moments_central-2-0,moments_central-2-1,moments_central-2-2,moments_central-2-3,moments_central-3-0,moments_central-3-1,moments_central-3-2,moments_central-3-3,bbox-0,bbox-1,bbox-2,bbox-3,area,convex_area,filled_area,major_axis_length,minor_axis_length,orientation,perimeter,centroid_y,centroid_x,solidity,cell_vol_vox_downstream,cell_vol_fl_downstream,2d_label_count,min_t,max_t,lifespan,age,frames_till_gone,elongation,Dia_Ph3_corrected_mean,Dia_Ph3_raw_sum,Dia_Ph3_corrected_amount,Dia_Ph3_corrected_concentration,max_frame_pos,file,selection_subset,position,directory,velocity_pixel,velocity_um,time_minutes,time_hours,will_divide,daughter_disappears_before_division,disappears_before_division,index,Dia_Ph3_amount_manualBkgr,Dia_Ph3_manualBkgr_bkgrVal_mean,Dia_Ph3_manualBkgr_bkgrVal_median,Dia_Ph3_manualBkgr_bkgrVal_q05,Dia_Ph3_manualBkgr_bkgrVal_q25,Dia_Ph3_manualBkgr_bkgrVal_q75,Dia_Ph3_manualBkgr_bkgrVal_q95,Dia_Ph3_mean_manualBkgr,bbox,centroid,inertia_tensor,inertia_tensor_eigvals,label,local_centroid,moments,moments_central,moments_normalized,num_objects,end_of_cell_cycle_frame_i,time_seconds_rel,Cell_ID_rel,is_cell_dead_rel,is_cell_excluded_rel,x_centroid_rel,y_centroid_rel,was_manually_edited_rel,cell_cycle_stage_rel,generation_num_rel,relative_ID_rel,relationship_rel,emerg_frame_i_rel,division_frame_i_rel,is_history_known_rel,corrected_assignment_rel,cell_area_pxl_rel,cell_area_um2_rel,cell_vol_vox_rel,cell_vol_fl_rel,Dia_Ph3_mean_rel,Dia_Ph3_sum_rel,Dia_Ph3_amount_autoBkgr_rel,Dia_Ph3_amount_dataPrepBkgr_rel,Dia_Ph3_concentration_autoBkgr_from_vol_vox_rel,Dia_Ph3_concentration_dataPrepBkgr_from_vol_vox_rel,Dia_Ph3_concentration_autoBkgr_from_vol_fl_rel,Dia_Ph3_concentration_dataPrepBkgr_from_vol_fl_rel,Dia_Ph3_median_rel,Dia_Ph3_min_rel,Dia_Ph3_max_rel,Dia_Ph3_q25_rel,Dia_Ph3_q75_rel,Dia_Ph3_q05_rel,Dia_Ph3_q95_rel,Dia_Ph3_autoBkgr_bkgrVal_median_rel,Dia_Ph3_autoBkgr_bkgrVal_mean_rel,Dia_Ph3_au

# ***Assigning delta vol normalised to cell vol for both data*** 

In [10]:
nutrient_switch_overall_df['delta_vol_norm_cellvol'] = (nutrient_switch_overall_df['vol_added_since_last_frame']/nutrient_switch_overall_df['combined_mother_bud_volume'])
steady_state_overall_df['delta_vol_norm_cellvol'] = (steady_state_overall_df['vol_added_since_last_frame']/steady_state_overall_df['sys_vol_at_frame_inc_G1_cells'])
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 400)
# display(steady_state_overall_df.head(10))
# display(nutrient_switch_overall_df.head(5))

,Unnamed: 0.1,index,Unnamed: 0,generation_num,Position,Cell_ID,growthmedium,frame_i,cell_cycle_stage,relative_ID,relationship,cell_vol_fl,emerg_frame_i,division_frame_i,len_G1,date,Strain,Birth frame,Birth vol fl,last_G1_frame,size_G1_end,vol_added_G1,Div frame,Div vol fl,len_S,mother_size_emerg,mother_size_div,vol_added_S_mother,bud_size_emerg,bud_size_div,vol_added_S_bud,sys_size_emerg,sys_size_div,vol_added_S_tot,total vol added,time_seconds,is_cell_dead,complete_phase,lenG1_mins,G1_growthrate_flpermin,delta_vol_fl,delta_time_mins,abs_growth_rate,Birth_or_div_vol_fl,rel_vol_atframe,sys_vol_atframe,delta_vol_sincePhaseStart_mother,delta_vol_sincePhaseStart_sys,norm_delta_vol_sincePhaseStart_mother,norm_delta_vol_sincePhaseStart_sys,bud_vol_atframe/mother_vol_atframe,delta_vol_sinceG1Start,norm_delta_vol_sinceG1Start,Daughterhood_binary,time_in_stage_mins,sys_vol_at_frame_inc_G1_cells,f_pos_cell_id,complete_cycle,frames_since_cycle_start,vol_added_since_last_frame,delta_vol_norm_cellvol
0,0,0.0,0.0,0.0,Position_1,1,SCD,0,S,4.0,bud,58.256695,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,False,NaN,NaN,NaN,18,NaN,NaN,92.927348,151.184043,NaN,NaN,NaN,NaN,1.595136,NaN,NaN,0,0.0,151.184043,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,0.0,NaN,NaN
1,1,1.0,1.0,0.0,Position_1,1,SCD,1,S,4.0,bud,60.404339,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.0,0,False,NaN,NaN,NaN,18,NaN,NaN,93.706004,154.110342,NaN,NaN,NaN,NaN,1.551312,NaN,NaN,0,3.0,154.110342,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,1.0,2.926300,0.018988
2,2,2.0,2.0,0.0,Position_1,1,SCD,2,S,4.0,bud,61.139191,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,360.0,0,False,NaN,NaN,NaN,18,NaN,NaN,94.112864,155.252054,NaN,NaN,NaN,NaN,1.539321,NaN,NaN,0,6.0,155.252054,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,2.0,1.141712,0.007354
3,3,3.0,3.0,0.0,Position_1,1,SCD,3,S,4.0,bud,62.500215,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,540.0,0,False,NaN,NaN,NaN,18,NaN,NaN,95.679685,158.179900,NaN,NaN,NaN,NaN,1.530870,NaN,NaN,0,9.0,158.179900,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,3.0,2.927846,0.018510
4,4,4.0,4.0,0.0,Position_1,1,SCD,4,S,4.0,bud,63.211776,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,720.0,0,False,NaN,NaN,NaN,18,NaN,NaN,97.265445,160.477221,NaN,NaN,NaN,NaN,1.538723,NaN,NaN,0,12.0,160.477221,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,4.0,2.297321,0.014316
5,5,5.0,5.0,0.0,Position_1,1,SCD,5,S,4.0,bud,63.341337,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,900.0,0,False,NaN,NaN,NaN,18,NaN,NaN,99.942398,163.283735,NaN,NaN,NaN,NaN,1.577838,NaN,NaN,0,15.0,163.283735,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,5.0,2.806514,0.017188
6,6,6.0,6.0,0.0,Position_1,1,SCD,6,S,4.0,bud,64.975784,-1.0,-1.0,NaN,20012021,MMY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1080.0,0,False,NaN,NaN,NaN,18,NaN,NaN,102.084921,167.060705,NaN,NaN,NaN,NaN,1.571123,NaN,NaN,0,18.0,167.060705,SCD_MMY_Position_1_Cell_1_Gen_0_Date_20012021,0.0,6.0,3.776970,0.022608
7,7,7.0,7.0,1.0,Position_1,1,SCD,7,G1,4.0,mother,67.499547,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1260.0,0,True,39.0,0.496112,71.738164,111,0.64629,67.499547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1,0.0,67.499547,SCD_MMY_Position_1_Cell_1_Gen_1_Date_20012021,1.0,0.0,NaN,NaN
8,8,8.0,8.0,1.0,Position_1,1,SCD,8,G1,4.0,mother,67.411987,-1.0,7.0,13.0,20012021,MMY,7.0,67.499547,19.0,86.847911,19.348364,NaN,NaN,25.0,84.25881,91.526842,7.268032,0.596295,47.710869,47.114574,84.855105,139.237711,54.382606,73.73097,1440.0,0,Tru

,frame_i,time_seconds,Cell_ID,is_cell_dead,is_cell_excluded,x_centroid,y_centroid,was_manually_edited,cell_cycle_stage,generation_num,relative_ID,relationship,emerg_frame_i,division_frame_i,is_history_known,corrected_assignment,cell_area_pxl,cell_area_um2,cell_vol_vox,cell_vol_fl,Dia_Ph3_mean,Dia_Ph3_sum,Dia_Ph3_amount_autoBkgr,Dia_Ph3_amount_dataPrepBkgr,Dia_Ph3_concentration_autoBkgr_from_vol_vox,Dia_Ph3_concentration_dataPrepBkgr_from_vol_vox,Dia_Ph3_concentration_autoBkgr_from_vol_fl,Dia_Ph3_concentration_dataPrepBkgr_from_vol_fl,Dia_Ph3_median,Dia_Ph3_min,Dia_Ph3_max,Dia_Ph3_q25,Dia_Ph3_q75,Dia_Ph3_q05,Dia_Ph3_q95,Dia_Ph3_autoBkgr_bkgrVal_median,Dia_Ph3_autoBkgr_bkgrVal_mean,Dia_Ph3_autoBkgr_bkgrVal_q75,Dia_Ph3_autoBkgr_bkgrVal_q25,Dia_Ph3_autoBkgr_bkgrVal_q95,Dia_Ph3_autoBkgr_bkgrVal_q05,Dia_Ph3_dataPrepBkgr_bkgrVal_median,Dia_Ph3_dataPrepBkgr_bkgrVal_mean,Dia_Ph3_dataPrepBkgr_bkgrVal_q75,Dia_Ph3_dataPrepBkgr_bkgrVal_q25,Dia_Ph3_dataPrepBkgr_bkgrVal_q95,Dia_Ph3_dataPrepBkgr_bkgrVal_q05,Dia_Ph3_CV,major_axis_length_gui,minor_axis_length_gui,inertia_tensor_eigvals-0,inertia_tensor_eigvals-1,equivalent_diameter,moments-0-0,moments-0-1,moments-0-2,moments-0-3,moments-1-0,moments-1-1,moments-1-2,moments-1-3,moments-2-0,moments-2-1,moments-2-2,moments-2-3,moments-3-0,moments-3-1,moments-3-2,moments-3-3,area_gui,solidity_gui,extent,inertia_tensor-0-0,inertia_tensor-0-1,inertia_tensor-1-0,inertia_tensor-1-1,filled_area_gui,centroid-0,centroid-1,bbox_area,local_centroid-0,local_centroid-1,convex_area_gui,euler_number,moments_normalized-0-0,moments_normalized-0-1,moments_normalized-0-2,moments_normalized-0-3,moments_normalized-1-0,moments_normalized-1-1,moments_normalized-1-2,moments_normalized-1-3,moments_normalized-2-0,moments_normalized-2-1,moments_normalized-2-2,moments_normalized-2-3,moments_normalized-3-0,moments_normalized-3-1,moments_normalized-3-2,moments_normalized-3-3,moments_central-0-0,moments_central-0-1,moments_central-0-2,moments_central-0-3,moments_central-1-0,moments_central-1-1,moments_central-1-2,moments_central-1-3,moments_central-2-0,moments_central-2-1,moments_central-2-2,moments_central-2-3,moments_central-3-0,moments_central-3-1,moments_central-3-2,moments_central-3-3,bbox-0,bbox-1,bbox-2,bbox-3,area,convex_area,filled_area,major_axis_length,minor_axis_length,orientation,perimeter,centroid_y,centroid_x,solidity,cell_vol_vox_downstream,cell_vol_fl_downstream,2d_label_count,min_t,max_t,lifespan,age,frames_till_gone,elongation,Dia_Ph3_corrected_mean,Dia_Ph3_raw_sum,Dia_Ph3_corrected_amount,Dia_Ph3_corrected_concentration,max_frame_pos,file,selection_subset,position,directory,velocity_pixel,velocity_um,time_minutes,time_hours,will_divide,daughter_disappears_before_division,disappears_before_division,index,Dia_Ph3_amount_manualBkgr,Dia_Ph3_manualBkgr_bkgrVal_mean,Dia_Ph3_manualBkgr_bkgrVal_median,Dia_Ph3_manualBkgr_bkgrVal_q05,Dia_Ph3_manualBkgr_bkgrVal_q25,Dia_Ph3_manualBkgr_bkgrVal_q75,Dia_Ph3_manualBkgr_bkgrVal_q95,Dia_Ph3_mean_manualBkgr,bbox,centroid,inertia_tensor,inertia_tensor_eigvals,label,local_centroid,moments,moments_central,moments_normalized,num_objects,end_of_cell_cycle_frame_i,time_seconds_rel,Cell_ID_rel,is_cell_dead_rel,is_cell_excluded_rel,x_centroid_rel,y_centroid_rel,was_manually_edited_rel,cell_cycle_stage_rel,generation_num_rel,relative_ID_rel,relationship_rel,emerg_frame_i_rel,division_frame_i_rel,is_history_known_rel,corrected_assignment_rel,cell_area_pxl_rel,cell_area_um2_rel,cell_vol_vox_rel,cell_vol_fl_rel,Dia_Ph3_mean_rel,Dia_Ph3_sum_rel,Dia_Ph3_amount_autoBkgr_rel,Dia_Ph3_amount_dataPrepBkgr_rel,Dia_Ph3_concentration_autoBkgr_from_vol_vox_rel,Dia_Ph3_concentration_dataPrepBkgr_from_vol_vox_rel,Dia_Ph3_concentration_autoBkgr_from_vol_fl_rel,Dia_Ph3_concentration_dataPrepBkgr_from_vol_fl_rel,Dia_Ph3_median_rel,Dia_Ph3_min_rel,Dia_Ph3_max_rel,Dia_Ph3_q25_rel,Dia_Ph3_q75_rel,Dia_Ph3_q05_rel,Dia_Ph3_q95_rel,Dia_Ph3_autoBkgr_bkgrVal_median_rel,Dia_Ph3_autoBkgr_bkgrVal_mean_rel,Dia_Ph3_au

In [11]:

# print(nutrient_switch_overall_df[nutrient_switch_overall_df['rep'] == 3].post_switch_gen.unique())
# print(nutrient_switch_overall_df[nutrient_switch_overall_df['rep'] == 2].post_switch_gen.unique())
# print(nutrient_switch_overall_df[nutrient_switch_overall_df['rep'] == 1].post_switch_gen.unique())

nutrient_switch_overall_df.loc[(((nutrient_switch_overall_df['rep'] == 1) | (nutrient_switch_overall_df['rep'] == 2))
                                & ((nutrient_switch_overall_df['post_switch_gen'] == '0') | (nutrient_switch_overall_df['post_switch_gen'] == 0))), 
                               'post_switch_gen'] = '0.0'
nutrient_switch_overall_df.loc[(((nutrient_switch_overall_df['rep'] == 1) | (nutrient_switch_overall_df['rep'] == 2))
                                & ((nutrient_switch_overall_df['post_switch_gen'] == '1') | (nutrient_switch_overall_df['post_switch_gen'] == 1))), 
                               'post_switch_gen'] = '1.0'
nutrient_switch_overall_df.loc[(((nutrient_switch_overall_df['rep'] == 1) | (nutrient_switch_overall_df['rep'] == 2))
                                & ((nutrient_switch_overall_df['post_switch_gen'] == '2') | (nutrient_switch_overall_df['post_switch_gen'] == 2))), 
                               'post_switch_gen'] = '2.0'
nutrient_switch_overall_df.loc[(((nutrient_switch_overall_df['rep'] == 1) | (nutrient_switch_overall_df['rep'] == 2))
                                & ((nutrient_switch_overall_df['post_switch_gen'] == '3') | (nutrient_switch_overall_df['post_switch_gen'] == 3))), 
                               'post_switch_gen'] = '3.0'
nutrient_switch_overall_df.loc[(((nutrient_switch_overall_df['rep'] == 1) | (nutrient_switch_overall_df['rep'] == 2))
                                & ((nutrient_switch_overall_df['post_switch_gen'] == '4') | (nutrient_switch_overall_df['post_switch_gen'] == 4))), 
                               'post_switch_gen'] = '4.0'
nutrient_switch_overall_df.loc[(((nutrient_switch_overall_df['rep'] == 1) | (nutrient_switch_overall_df['rep'] == 2))
                                & ((nutrient_switch_overall_df['post_switch_gen'] == '5') | (nutrient_switch_overall_df['post_switch_gen'] == 5))), 
                               'post_switch_gen'] = '5.0'

# print(nutrient_switch_overall_df[nutrient_switch_overall_df['rep'] == 2].post_switch_gen.unique())
# print(nutrient_switch_overall_df[nutrient_switch_overall_df['rep'] == 1].post_switch_gen.unique())


[nan 'switch_gen' '1.0' '2.0' '3.0' '4.0' '0.0' '5.0']
[nan 'switch_gen' '1' '2' '3' '4' '0' 0.0 1.0 2.0 '5']
[nan 'switch_gen' '1.0' '2.0' '3.0' '0.0' '4.0' 2.0 3.0 0.0 1.0 '5.0' 4.0]
[nan 'switch_gen' '1.0' '2.0' '3.0' '4.0' '0.0' '5.0']
[nan 'switch_gen' '1.0' '2.0' '3.0' '0.0' '4.0' '5.0']


In [12]:
nutrient_switch_overall_df.to_csv(r"Y:\test_dfs_for_final_code\normalnutrientswitch_jupyter_overall_df_with_rel_switch_and_wave_info.csv")
steady_state_overall_df.to_csv(r"Y:\test_dfs_for_final_code\steadystate_analysed_fornutswitch_jupyter_overall_df_with_rel_switch_and_wave_info.csv")